# 03 — Bridging the Omics Layers

**Network Medicine Workshop · Kidney Disease · Part 3 of 3**

This is the payoff notebook. So far we've shown (separately) that your DE proteins form a connected module in the PPI network, and your DE metabolites form a connected module in the metabolite network. The real network-medicine question is: **do these two modules talk to each other?**

To answer that, we need a bridge between "gene/protein space" and "metabolite space" — KEGG compounds don't sit in the PPI network, but the enzymes that produce/consume them do. So:

1. Build a **metabolite → gene bridge** via KEGG (compound → enzyme (EC) → gene)
2. Merge it with your PPI network into one combined graph
3. Find the **shortest paths** between your PPI/protein module and your metabolite-linked genes
4. Pull out the genes sitting *on* those shortest paths — the "bridge module" — and enrich them
5. Check whether any bridge genes are known kidney-disease genes (Open Targets) — the "is this a real, actionable axis" check

**Timing: ~20 min.** Requires Notebooks 1 and 2 to have already run.

**A note on scale:** your networks can have up to ~20,000 nodes. The one thing to avoid at that scale is all-pairs shortest paths (`O(V^3)`), so instead we do a **multi-source Dijkstra from your (much smaller) protein module** using `scipy.sparse.csgraph`, which is vectorized and fast even at this size.


## Setup — reload from Notebooks 1 & 2

In [ ]:
!pip install -q scipy networkx gprofiler-official requests


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, pickle, time
import pandas as pd
import numpy as np
import networkx as nx

BASE_DIR = "/content/drive/MyDrive/network_medicine_workshop"
PROC_DIR = os.path.join(BASE_DIR, "processed")

PRIMARY_GENE_LAYER = "ppi"   # the protein/gene network we bridge INTO

with open(os.path.join(PROC_DIR, "modules.pkl"), "rb") as f:
    modules = pickle.load(f)

graphs = {}
for layer in ["ppi", "transcriptome", "metabolite"]:
    with open(os.path.join(PROC_DIR, f"graph_{layer}.pkl"), "rb") as f:
        graphs[layer] = pickle.load(f)

metabs_matched = pd.read_csv(os.path.join(PROC_DIR, "metabolites_matched.csv"), dtype={"kegg_id": str})
metab_ids = set(metabs_matched["kegg_id"].dropna().unique()) & set(graphs["metabolite"].nodes())

# the connected PPI module derived from your proteomics data (Notebook 2)
ppi_module_nodes = modules[PRIMARY_GENE_LAYER]["seed_nodes"]
print(f"Protein module (from proteomics, in the {PRIMARY_GENE_LAYER} network): {len(ppi_module_nodes)} nodes")
print(f"DE metabolites present in metabolite network: {len(metab_ids)}")


## Step 1 — Load the metabolite → gene bridge network

Rather than a bridge that's tied to one specific metabolite list, we use a **general-purpose bipartite network**: every KEGG compound linked to the human genes encoding the enzymes that act on it. This is precomputed **once** from KEGG's bulk `link` endpoint (two whole-database requests, joined locally — see `build_metabolite_gene_bridge.py` provided alongside these notebooks) rather than queried per-compound, so it finishes in seconds and — importantly — isn't scoped to your current metabolite list at all. Swap in a different `DE_metabolites.csv` in Notebook 1 and this same bridge file still works; no re-querying KEGG.

Run the prep script once, drop the resulting `metabolite_gene_bridge.csv` into your Drive's `networks/` folder, and this cell just loads + subsets it to whichever metabolites are in play right now. If the file's missing, it falls back to the old slow per-compound live-query path (with a warning) so the notebook still works standalone.


In [ ]:
import requests

BRIDGE_CSV = os.path.join(BASE_DIR, "networks", "metabolite_gene_bridge.csv")

if os.path.exists(BRIDGE_CSV):
    full_bridge_df = pd.read_csv(BRIDGE_CSV, dtype=str)
    print(f"Loaded general-purpose bridge network: {len(full_bridge_df)} edges, "
          f"{full_bridge_df['kegg_id'].nunique()} compounds, {full_bridge_df['ncbi_gene_id'].nunique()} genes")

    # subset to whichever metabolites are actually in play for THIS run
    relevant = full_bridge_df[full_bridge_df["kegg_id"].isin(metab_ids)]
    bridge_cache = relevant.groupby("kegg_id")["ncbi_gene_id"].apply(set).to_dict()
    print(f"Subset to your {len(metab_ids)} current metabolites: "
          f"{len(bridge_cache)} have >=1 linked gene")
else:
    print("[warn] networks/metabolite_gene_bridge.csv not found — falling back to live per-compound "
          "KEGG queries (slow; run build_metabolite_gene_bridge.py ahead of time next time).")

    def kegg_link(source_db, target_id):
        url = f"https://rest.kegg.jp/link/{source_db}/{target_id}"
        r = requests.get(url, timeout=30)
        if r.status_code != 200 or not r.text.strip():
            return []
        pairs = [line.split("\t") for line in r.text.strip().split("\n")]
        return [p[1] for p in pairs if len(p) == 2]

    def genes_for_compound(cid):
        genes = set()
        try:
            for ec in kegg_link("enzyme", f"cpd:{cid}"):
                for g in kegg_link("hsa", ec):
                    genes.add(g.replace("hsa:", ""))
        except Exception as e:
            print(f"  [warn] {cid}: {e}")
        return genes

    t0 = time.time()
    bridge_cache = {}
    for i, cid in enumerate(metab_ids):
        bridge_cache[cid] = genes_for_compound(cid)
        if (i + 1) % 25 == 0:
            print(f"  ...{i+1}/{len(metab_ids)}  ({time.time()-t0:.0f}s elapsed)")
    print(f"Done in {time.time()-t0:.0f}s")

n_with_genes = sum(1 for cid in metab_ids if bridge_cache.get(cid))
print(f"{n_with_genes}/{len(metab_ids)} metabolites linked to >=1 gene.")


## Step 2 — Merge into one combined graph, find shortest paths

We build a single graph: the PPI network, plus a bipartite layer of `metabolite –– gene` bridge edges. Then, instead of computing distances one pair at a time, we do **one multi-source Dijkstra call** from all protein-module nodes simultaneously (`scipy.sparse.csgraph.dijkstra`), and read off the distance to every metabolite-linked gene at once. This is the part that has to be efficient at 20k nodes, and this approach is.


In [ ]:
from scipy.sparse.csgraph import dijkstra

G_gene = graphs[PRIMARY_GENE_LAYER].copy()

# metabolite-linked genes: union of all genes bridged from any DE metabolite, restricted to
# genes that are actually in the PPI network (otherwise there's nothing to path-find to)
metab_linked_genes = set()
metab_to_genes = {}
for cid in metab_ids:
    genes_here = bridge_cache.get(cid, set()) & set(G_gene.nodes())
    if genes_here:
        metab_to_genes[cid] = genes_here
        metab_linked_genes |= genes_here

print(f"{len(metab_linked_genes)} distinct metabolite-linked genes present in the {PRIMARY_GENE_LAYER} network, "
      f"covering {len(metab_to_genes)}/{len(metab_ids)} metabolites")

# Build sparse adjacency + node index maps
nodes = list(G_gene.nodes())
node_idx = {n: i for i, n in enumerate(nodes)}
A = nx.to_scipy_sparse_array(G_gene, nodelist=nodes, weight=None, format="csr")

ppi_seed_idx = [node_idx[n] for n in ppi_module_nodes if n in node_idx]
print(f"Running multi-source Dijkstra from {len(ppi_seed_idx)} protein-module seed nodes over "
      f"{len(nodes):,} nodes / {G_gene.number_of_edges():,} edges...")

t0 = time.time()
dist_matrix = dijkstra(csgraph=A, directed=False, indices=ppi_seed_idx, unweighted=True)
# dist_matrix shape: (n_seeds, n_nodes) — take the min over seeds = distance from the *nearest* module node
min_dist_from_ppi_module = dist_matrix.min(axis=0)
print(f"Done in {time.time()-t0:.1f}s")


In [ ]:
# Distance from each metabolite-linked gene to the nearest protein-module node
records = []
for cid, genes_here in metab_to_genes.items():
    for g in genes_here:
        d = min_dist_from_ppi_module[node_idx[g]]
        records.append({"kegg_id": cid, "bridge_gene": g, "distance_to_ppi_module": d})

bridge_df = pd.DataFrame(records)
bridge_df = bridge_df[np.isfinite(bridge_df["distance_to_ppi_module"])]  # drop unreachable
bridge_df = bridge_df.sort_values("distance_to_ppi_module")

print("Distribution of shortest-path distances (metabolite-linked gene -> nearest protein-module node):")
print(bridge_df["distance_to_ppi_module"].value_counts().sort_index())
bridge_df.head(15)


**How to read this:** a distance of 1 means a metabolite-linked gene is a *direct interactor* of one of your DE proteins — about as tight a mechanistic link as network data can offer. Distance 2–3 still suggests a plausible shared pathway/complex. Longer distances (or "unreachable") suggest that particular metabolite's enzymes aren't operating anywhere near your proteomic signal — that's a real negative result, not a bug, and worth noting to the group.

Compare this observed distribution to what you'd expect from two random gene sets of the same sizes in this network if you want a formal significance statement — the degree-preserving permutation approach from Notebook 4 can be adapted for exactly this.


## Step 3 — Extract the bridge module

We reconstruct the actual shortest-path node sequences for the closest metabolite–protein connections (cheap — only doing this for a handful of pairs, not all of them) and take the union of nodes on those paths. This "bridge module" is the concrete, inspectable answer to *"how are the proteome and metabolome connected, mechanistically?"*


In [ ]:
N_CLOSEST = 25   # how many closest metabolite-gene connections to trace explicitly

closest = bridge_df.head(N_CLOSEST)
bridge_module_nodes = set()
path_records = []

for _, row in closest.iterrows():
    gene = row["bridge_gene"]
    # nearest protein-module node to this gene (search over all seeds, cheap for a handful of genes)
    best_path, best_len = None, None
    for seed in ppi_module_nodes:
        if seed not in G_gene or gene not in G_gene:
            continue
        try:
            p = nx.shortest_path(G_gene, source=seed, target=gene)
            if best_len is None or len(p) < best_len:
                best_path, best_len = p, len(p)
        except nx.NetworkXNoPath:
            continue
    if best_path:
        bridge_module_nodes.update(best_path)
        path_records.append({
            "kegg_id": row["kegg_id"], "bridge_gene": gene,
            "ppi_module_node": best_path[0], "path_length": len(best_path) - 1,
            "path": " -> ".join(best_path)
        })

path_table = pd.DataFrame(path_records).sort_values("path_length")
print(f"Bridge module: {len(bridge_module_nodes)} unique genes across {len(path_table)} traced paths")
path_table.head(15)


## Step 4 — Enrichment of the bridge module

In [ ]:
from gprofiler import GProfiler

gp = GProfiler(return_dataframe=True)

if len(bridge_module_nodes) >= 3:
    bridge_enrichment = gp.profile(
        organism="hsapiens", query=list(bridge_module_nodes), sources=["GO:BP", "KEGG", "REAC"],
    ).sort_values("p_value")
    display(bridge_enrichment[["source", "name", "p_value", "term_size", "intersection_size"]].head(15))
    bridge_enrichment.to_csv(os.path.join(PROC_DIR, "bridge_module_enrichment.csv"), index=False)
else:
    print("Bridge module too small for enrichment — try increasing N_CLOSEST above.")
    bridge_enrichment = pd.DataFrame()


## Step 5 — Is any of this targetable? (Open Targets)

Last check: do any of the bridge-module genes already have known associations with kidney disease — and by extension, known drugs? We query the [Open Targets Platform](https://platform.opentargets.org/) public API (no key required). First we resolve a disease name to its EFO ID, then pull associated targets, then intersect with our bridge module.


In [ ]:
OT_API = "https://api.platform.opentargets.org/api/v4/graphql"
DISEASE_QUERY = "chronic kidney disease"   # change to match your phenotype of interest

search_query = """
query searchDisease($q: String!) {
  search(queryString: $q, entityNames: ["disease"], page: {index: 0, size: 5}) {
    hits { id name entity }
  }
}
"""

try:
    r = requests.post(OT_API, json={"query": search_query, "variables": {"q": DISEASE_QUERY}}, timeout=30)
    r.raise_for_status()
    hits = r.json()["data"]["search"]["hits"]
    for h in hits:
        print(h["id"], "-", h["name"])
except Exception as e:
    print(f"[warn] Open Targets search failed: {e}")
    hits = []


In [ ]:
# Pick the EFO ID that best matches your phenotype from the printed list above
EFO_ID = hits[0]["id"] if hits else None
print("Using disease ID:", EFO_ID)

assoc_query = """
query diseaseTargets($efoId: String!, $size: Int!) {
  disease(efoId: $efoId) {
    associatedTargets(page: {index: 0, size: $size}) {
      rows {
        target { id approvedSymbol }
        score
      }
    }
  }
}
"""

disease_gene_symbols = set()
if EFO_ID:
    try:
        r = requests.post(OT_API, json={"query": assoc_query,
                                          "variables": {"efoId": EFO_ID, "size": 500}}, timeout=30)
        r.raise_for_status()
        rows = r.json()["data"]["disease"]["associatedTargets"]["rows"]
        disease_gene_symbols = {row["target"]["approvedSymbol"] for row in rows}
        print(f"Pulled {len(disease_gene_symbols)} disease-associated genes from Open Targets")
    except Exception as e:
        print(f"[warn] Open Targets association query failed: {e}")


In [ ]:
# Our bridge module is in NCBI Gene IDs; Open Targets returns gene symbols, so map back via
# the matched-symbol table saved in Notebook 1 (union of transcript + protein matches).
degs_matched = pd.read_csv(os.path.join(PROC_DIR, "degs_matched.csv"), dtype={"ncbi_gene_id": str})
proteins_matched = pd.read_csv(os.path.join(PROC_DIR, "proteins_matched.csv"), dtype={"ncbi_gene_id": str})
id_to_symbol = dict(zip(proteins_matched["ncbi_gene_id"], proteins_matched["matched_symbol"]))
id_to_symbol.update(dict(zip(degs_matched["ncbi_gene_id"], degs_matched["matched_symbol"])))

bridge_symbols = {id_to_symbol.get(g, g) for g in bridge_module_nodes}
overlap = bridge_symbols & disease_gene_symbols

print(f"Bridge module genes with a prior kidney-disease association: {len(overlap)} / {len(bridge_symbols)}")
if overlap:
    print(sorted(overlap))


**This is the slide to end on.** Independently-generated proteomic and metabolomic signals converge on a specific, connected set of genes in the interactome — and a subset of those genes already have documented kidney-disease associations (and, worth checking manually on the [Open Targets](https://platform.opentargets.org/) or [DrugBank](https://go.drugbank.com/) site, some may have approved or investigational compounds against them). That's the difference between "we found a correlation" and "we found a mechanism with a foothold for intervention."


## Step 6 — Final visualization

In [ ]:
import matplotlib.pyplot as plt

sub = G_gene.subgraph(bridge_module_nodes).copy()
pos = nx.spring_layout(sub, seed=0, k=0.6)

node_colors = []
for n in sub.nodes():
    if n in overlap or id_to_symbol.get(n, n) in overlap:
        node_colors.append("#C44E52")   # known disease gene within the bridge
    elif n in ppi_module_nodes:
        node_colors.append("#4C72B0")   # protein-module node (proteomics)
    elif n in metab_linked_genes:
        node_colors.append("#55A868")   # metabolite-linked gene
    else:
        node_colors.append("#8172B2")   # intermediate bridge node

plt.figure(figsize=(9, 9))
nx.draw_networkx_edges(sub, pos, alpha=0.3)
nx.draw_networkx_nodes(sub, pos, node_color=node_colors, node_size=250, alpha=0.9)
labels = {n: id_to_symbol.get(n, n) for n in sub.nodes()}
nx.draw_networkx_labels(sub, pos, labels=labels, font_size=7)

legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', label='Protein-module gene (proteomics)', markerfacecolor='#4C72B0', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Metabolite-linked gene', markerfacecolor='#55A868', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Intermediate bridge node', markerfacecolor='#8172B2', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Known kidney-disease gene', markerfacecolor='#C44E52', markersize=10),
]
plt.legend(handles=legend_elements, loc='upper left', fontsize=9)
plt.title("Cross-omics bridge module: proteome <-> metabolome, kidney disease")
plt.axis("off")
plt.savefig(os.path.join(PROC_DIR, "bridge_module.png"), dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Save the bridge module — Notebook 4 (optional) can use this as "your" module for disease comparisons
with open(os.path.join(PROC_DIR, "bridge_module.pkl"), "wb") as f:
    pickle.dump(bridge_module_nodes, f)


---
## Discussion prompts for the group

- Which pathways came up in the bridge-module enrichment — does it match a known kidney-injury program (e.g. mitochondrial/fatty-acid oxidation, complement/immune, fibrosis/ECM)?
- If you have spatial coordinates for the DEGs, where in the tissue does the bridge module light up? Does that change the interpretation?
- For genes flagged as known kidney-disease genes: are any of them existing drug targets? Would this suggest a repurposing candidate, or just validate the module as biologically real?
- What would you need to see to trust this result enough to prioritize it for follow-up (e.g. IHC validation, an orthogonal cohort)?

**Optional next step:** `04_disease_modules_optional.ipynb` picks this bridge module back up and asks how it relates, network-topologically, to two *other* diseases.
